In [ ]:
import numpy as np

def unified_bcgd(X, k, steps=200, innerW=1, innerH=1, 
                 alpha_W=1e-3, alpha_H=1e-3, method='gd', beta=0.9, 
                 is_nmf=True, M=None, lmbda=0.0):
    """
    Algoritmo 1: Unified Block-Coordinate Gradient Descent for Matrix Factorization
    Configurado por defecto para Proyecto C (Audio NMF).
    """
    m, n = X.shape
    
    # Manejo de la máscara M (si no se manda, es una matriz de unos)
    if M is None:
        M = np.ones((m, n))
        
    # Función de proyección (Punto 2.4)
    def proj(matrix):
        if is_nmf:
            return np.maximum(matrix, 0)
        return matrix

    # 1. Inicialización (Valores positivos pequeños, recomendación Parte 5.1)
    # Usamos 1/sqrt(k) para controlar la escala inicial y evitar explosiones
    W = np.random.rand(m, k) / np.sqrt(k)
    H = np.random.rand(k, n) / np.sqrt(k)
    
    # 2. Inicialización de velocidades
    vW = np.zeros((m, k))
    vH = np.zeros((k, n))
    
    loss_history = []
    
    # 3. Bucle principal
    for s in range(steps):
        
        # ----------------------------------------------------
        # BLOQUE W (Líneas 4 a 20)
        # ----------------------------------------------------
        for t in range(innerW):
            if method == 'nesterov':
                # Paso adelantado (Lookahead)
                W_look = W - alpha_W * beta * vW
                R_look = M * ((W_look @ H) - X)
                gW_look = R_look @ H.T + lmbda * W_look
                
                # Actualización Nesterov
                vW = beta * vW + gW_look
                W = W - alpha_W * vW
                
            else:
                # Cálculo normal de residual y gradiente
                R = M * ((W @ H) - X)
                gW = R @ H.T + lmbda * W
                
                if method == 'gd':
                    W = W - alpha_W * gW
                elif method == 'momentum':
                    vW = beta * vW + gW
                    W = W - alpha_W * vW
            
            # Proyección (Punto 2.4)
            W = proj(W)
            
        # ----------------------------------------------------
        # BLOQUE H (Líneas 21 a 26)
        # ----------------------------------------------------
        for t in range(innerH):
            if method == 'nesterov':
                # Paso adelantado (Lookahead)
                H_look = H - alpha_H * beta * vH
                R_look = M * ((W @ H_look) - X)
                gH_look = W.T @ R_look + lmbda * H_look
                
                # Actualización Nesterov
                vH = beta * vH + gH_look
                H = H - alpha_H * vH
                
            else:
                # Cálculo normal de residual y gradiente
                R = M * ((W @ H) - X)
                gH = W.T @ R + lmbda * H
                
                if method == 'gd':
                    H = H - alpha_H * gH
                elif method == 'momentum':
                    vH = beta * vH + gH
                    H = H - alpha_H * vH
            
            # Proyección (Punto 2.4)
            H = proj(H)
            
        # ----------------------------------------------------
        # CÁLCULO DE LA PÉRDIDA (Línea 27)
        # ----------------------------------------------------
        R_final = M * ((W @ H) - X)
        loss = 0.5 * np.sum(R_final ** 2) + (lmbda / 2) * (np.sum(W ** 2) + np.sum(H ** 2))
        loss_history.append(loss)
        
        # Opcional: Imprimir progreso cada 50 iteraciones para no saturar la consola
        if (s + 1) % 50 == 0 or s == 0:
            print(f"Iteración {s + 1:4d}/{steps} | Loss: {loss:.4f}")
            
    return W, H, loss_history

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Asumimos que la función unified_bcgd() que armamos en el paso anterior 
# ya está cargada en memoria.

# 1. Crear una matriz X de prueba (simulando un espectrograma STFT)
# En tu script final, aquí usarás S_mag = np.abs(librosa.stft(y, ...))
np.random.seed(42)
m, n = 513, 216  # Dimensiones típicas de tu STFT del MUSAN EDA
k_true = 10
# Creamos datos sintéticos no negativos
W_true = np.abs(np.random.randn(m, k_true))
H_true = np.abs(np.random.randn(k_true, n))
X_test = W_true @ H_true + np.abs(np.random.randn(m, n) * 0.1)

# 2. Parámetros por defecto para el Proyecto C (Audio NMF)
k_model = 10
steps = 300
alpha = 1e-3
beta = 0.9

# Diccionario para guardar los historiales
history = {}
methods = ['gd', 'momentum', 'nesterov']
colors = {'gd': '#4C72B0', 'momentum': '#DD8452', 'nesterov': '#55A868'}

# 3. Bucle de evaluación
print("Iniciando comparación de optimizadores...")
for method in methods:
    print(f"-> Entrenando con {method.upper()}...")
    
    # TRUCO VITAL: Fijar la semilla ANTES de cada ejecución.
    # Esto garantiza que los 3 métodos arranquen desde los mismos W y H exactos.
    np.random.seed(123)
    
    W_out, H_out, loss_hist = unified_bcgd(
        X=X_test, k=k_model, steps=steps, 
        innerW=1, innerH=1, 
        alpha_W=alpha, alpha_H=alpha, 
        method=method, beta=beta, 
        is_nmf=True, lmbda=0.0
    )
    history[method] = loss_hist

# 4. Graficar (Requisito 8.1 y 10.2)
plt.figure(figsize=(10, 6))
for method in methods:
    plt.plot(history[method], label=method.upper(), color=colors[method], linewidth=2)

plt.title('Comparación de Optimizadores: Loss vs Iteration (Audio NMF)', fontsize=14, fontweight='bold')
plt.xlabel('Iteraciones', fontsize=12)
plt.ylabel('Función de Pérdida f(W,H)', fontsize=12)
plt.yscale('log') # La escala logarítmica permite ver mejor la convergencia asintótica
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()